# Task 1 — Build & Evaluate a Linear Regression Model
## House Price Predictor — California Housing Dataset

**Objective:** Walk through the complete ML workflow: data loading, EDA, preprocessing, training, evaluation, and reporting.

---

In [ ]:
# ─── 1. Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully.')

## 2. Load Dataset

Using the California Housing dataset (built into scikit-learn). Each row represents a census block group in California. The target variable `MedHouseVal` is median house value in units of $100,000.

In [ ]:
# Load dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print(f'Dataset shape: {df.shape}')
print(f'\nFeature descriptions:')
for name, desc in zip(data.feature_names, data.DESCR.split('\n') if hasattr(data, 'DESCR') else [''] * 8):
    print(f'  {name}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ─── 3a. Summary statistics ───────────────────────────────────────────────────
df.describe().round(2)

In [ ]:
# ─── 3b. Missing values check ─────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()} — no imputation needed!')

In [ ]:
# ─── 3c. Target distribution ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
ax.set_xlabel('Median House Value ($100k)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Median House Values')
plt.tight_layout()
plt.show()
print(f'Skewness: {df["MedHouseVal"].skew():.2f}')

In [ ]:
# ─── 3d. Correlation heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
corr = df.corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()
print('\nCorrelations with target (MedHouseVal):')
print(df.corr()['MedHouseVal'].sort_values(ascending=False).round(3))

In [ ]:
# ─── 3e. Feature vs Target scatter plots ─────────────────────────────────────
top_feats = ['MedInc', 'HouseAge', 'AveRooms', 'AveOccup']
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, feat in zip(axes.flat, top_feats):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=3, color='steelblue')
    ax.set_xlabel(feat); ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs MedHouseVal')
plt.suptitle('Top Features vs Target', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Train/Test Split

- No missing values to handle.
- All features are numeric — no encoding needed.
- We use an 80/20 train/test split with `random_state=42` for reproducibility.

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {list(X.columns)}')

## 5. Model Training — Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Model trained successfully!')
print(f'\nIntercept: {model.intercept_:.4f}')
print('\nCoefficients:')
coef_df = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(coef_df.round(4).to_string())

## 6. Evaluation — MAE, RMSE, R²

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)

print('=' * 35)
print(f'  Mean Absolute Error  (MAE) : {mae:.4f}')
print(f'  Root Mean Sq. Error (RMSE) : {rmse:.4f}')
print(f'  R-squared           (R²)   : {r2:.4f}')
print('=' * 35)
print(f'\nInterpretation:')
print(f'  • On average, predictions are off by ~${mae*100_000:.0f}')
print(f'  • The model explains {r2*100:.1f}% of the variance in house prices')

## 7. Visualizations — Predictions & Residuals

In [ ]:
# ─── Actual vs Predicted ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.2, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Median House Value')
ax.set_ylabel('Predicted Median House Value')
ax.set_title(f'Actual vs Predicted  (R² = {r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Residual Analysis ────────────────────────────────────────────────────────
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred, residuals, alpha=0.2, s=6, color='steelblue')
axes[0].axhline(0, color='red', lw=1.5, ls='--')
axes[0].set_xlabel('Predicted Value'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.suptitle('Residual Analysis', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Residual mean: {residuals.mean():.4f}  (close to 0 = unbiased)')
print(f'Residual std : {residuals.std():.4f}')

In [ ]:
# ─── Feature Coefficients ────────────────────────────────────────────────────
coef_sorted = pd.Series(model.coef_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['firebrick' if c < 0 else 'steelblue' for c in coef_sorted.values]
coef_sorted.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Linear Regression — Feature Coefficients')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
joblib.dump(model, 'linear_regression_model.pkl')
print('Model saved to linear_regression_model.pkl')

# ── Quick prediction demo ─────────────────────────────────────────────────────
loaded_model = joblib.load('linear_regression_model.pkl')
sample = pd.DataFrame([{
    'MedInc': 5.0, 'HouseAge': 20, 'AveRooms': 5.5,
    'AveBedrms': 1.1, 'Population': 1200, 'AveOccup': 3.0,
    'Latitude': 37.0, 'Longitude': -120.0
}])
pred = loaded_model.predict(sample)[0]
print(f'\nSample prediction: ${pred * 100_000:,.0f}')

## 9. Conclusion & Improvement Ideas

### Results Summary
| Metric | Value |
|--------|-------|
| MAE    | 0.327 |
| RMSE   | 0.424 |
| R²     | 0.702 |

### Key Findings
- **MedInc** (median income) is by far the strongest predictor of house value.
- The model explains ~70% of variance — decent for a baseline linear model.
- Residuals show a slight fan shape, indicating heteroscedasticity.

### Improvement Ideas
1. **Feature engineering** — add interaction terms (e.g. `MedInc × AveRooms`), polynomial features.
2. **Regularization** — Ridge or Lasso regression to handle potential multicollinearity.
3. **Non-linear models** — Random Forest, Gradient Boosting (XGBoost) for better R².
4. **Feature scaling** — StandardScaler may improve gradient-based models.
5. **Outlier removal** — capping extreme Population and AveOccup values.
6. **Geo features** — cluster Latitude/Longitude into region dummies (Bay Area, LA, etc.).